# TIEGCM Parsl workflow

This notebook is the stand-alone companion to the Parsl workflow in `main.py` in this repository. This notebook is designed to be run directly on an HPC resource while the `main.py` in this workflow uses the `parsl_utils` to launch ensemble members from a central coordinating node (i.e. a laptop or the Parallel Works platform). This workflow simulates a typical TIEGCM data assimilating workflow with the following tasks:
1. configure Parsl and start Parsl monitoring to visualize workflow progress;
2. download TIEGCM container;
3. download and preprocess solar forcing;
4. set up siloed work directories for ensemble members;
5. launch and monitor the ensemble members;
6. post process the ensemble member output;
7. run data assimilation (or a DA placeholder); and
8. any clean up/restructuring necessary for running another DA cycle.

## Workflow visualization

There are three options for visualizing a Parsl workflow:
1. Manual "direct" launch of `parsl-visualize` before workflow runs;
2. Visualization launched as part of the workflow (i.e. self-launch); and
3. Offline visualization (i.e. for browsing previous Parsl workflows or starting visualization completely independently of the workflow).

Examples for all three are provided below but options \#1 and \#3 are commented out.

## Workflow parameters

The key customizable parameters for this workflow are defined immediately below. For a fully automated workflow (i.e. non-interactive workflow in `main.py`), these parameters are typically specified in the workflow launch form (and corresponding `.json` package for API launch) and then they make it to the command line launch of `main.py` on the head node of the cluster.

In [171]:
# Workflow parameters --- this cell is tagged parameters for Papermill
# as described at https://papermill.readthedocs.io/en/latest/usage-parameterize.html#jupyterlab-3-0

# We need the os library for shell things
import os
# Need time for sleep
import time
# Bootstrap installation info
install=False
install_from_scratch=False
conda_base_path="~/pw/software/.miniconda3n/"
conda_env_name="parsl"

# App workdir path info
param_log_dir='./parsl-app-logs'
param_work_dir_root='./tiegcm-work'
param_data_dir_root=param_work_dir_root+'/data'
# Prefix matches **singularity pull** => containers on DockerHub are prefixed with docker:// 
param_container_url='docker://parallelworks/tiegcm:latest'

# Set the default partition name if 'None' is specified - else assume it's coming from papermill
# and gets set from that argument
param_partition="c5a-8xlarge"

# default wallclock time limit of 1 hour
param_walltime_limit="01:00:00"

# Ensemble size and other solar forcing parameters
param_ens_size=10
param_range_limit=5
# Will not return successful F107 for "incomplete" days
# The delay is typically 1-2 days in the past.
# "Today" generally won't work and throws an error.
# "Yesterday" will generate a mean but no std. dev.
# The day before yesterday seems to work reliably.
# FORMAT: YYYY/MM/DD/HH:MM:SS
# tiegcm_res5.0_mareqx_smin_prim.nc is 2002 day 80 00:00:00 -> March 21
param_datetime_str="2023/09/21/00:00:00"
param_simulation_hours=2
# Mean and std dev of F107 solar param are updated
# in near real time based on param_date_str. These
# params are not needed.
#param_mean=75
#param_std_dev=1
#=================================
# Initial conditions
# `cold` starts use SOURCE in the .inp (namelist) and are
# not sensitive to the time in the actual source file.
# `warm` starts use the OUTPUT=[src_file,out_file] in the
# .inp (namelist) file and the date of the src file must
# match the date of the simulation start.
param_start_type='cold'
param_src_file='data/tiegcm_res5.0_data/tiegcm_res5.0_sepeqx_smax_prim.nc'

In [172]:
# Partition logic
try:
    if param_partition == "None":
        param_partition = !sinfo | grep '*' | awk '{print $1}' | sed 's/*//'
        param_partition = param_partition[0].strip() if param_partition else None
except NameError:
    raise RuntimeError("param_partition is not set. Papermill did not inject a value, and no default is defined.")


## Installs

There are two install options:
1. `install_from_scratch = True` documents the steps to build a particular environment
2. `install_from_scratch = False` is faster to reconstruct a Conda environment from an exported env file `.yaml` than to rebuild from scratch and promotes reproducibility.

The reconstruction command is kept active here since 
env files are distributed with this notebook. Once the command to reconstruct
the Conda environment has been run, you may need to tell
this notebook to use the kernel from that Conda environment
with the `Kernel > Change kernel...` option in the menu above.

In [173]:
# You don't need to rerun the install if the environment has already
# been built and selected as the kernel for this notebook.
if (install):
    if (install_from_scratch):
        
        # Currently there is a dependency bug with ipykernel and Python 3.13,
        # so pin to a different Python.
        ! conda create -y --name {conda_env_name} python=3.9
        
        # To use a Jupyter notebook with a
        # specific conda environment:
        ! conda install -y --name {conda_env_name} requests
        ! conda install -y --name {conda_env_name} ipykernel
        ! conda install -y --name {conda_env_name} -c anaconda jinja2
        
        # Additional packages for near-real time data streams:
        ! conda install -y --name {conda_env_name} psycopg2
        
        # Packages for writing model namelists:
        ! conda install -y --name {conda_env_name} scipy
        ! conda install -y --name {conda_env_name} pydantic
        
        # pip installs
        # Conda does not install monitoring, so use pip
        # Each Conda env has its own pip, so need to activate.
        # All pip installs need to happen after conda installs.
        ! source {conda_base_path}/etc/profile.d/conda.sh; conda activate {conda_env_name}; pip install --upgrade pip
        ! source {conda_base_path}/etc/profile.d/conda.sh; conda activate {conda_env_name}; pip install 'parsl[monitoring, visualization]'
        # Papermill is needed for automated launch of this notebook
        ! source {conda_base_path}/etc/profile.d/conda.sh; conda activate {conda_env_name}; pip install papermill
        
        # The environment was then exported with:
        ! conda env export --name {conda_env_name} > ./requirements/{conda_env_name}.yaml
    else:
        # You can rebuild the environment with:
        ! conda env update -f ./requirements/{conda_env_name}.yaml --name {conda_env_name}


## Imports

Based on the instructions in the [Parsl Tutorial](https://parsl.readthedocs.io/en/latest/1-parsl-introduction.html)

In [174]:
import os
import sys
import numpy as np
#import pandas as pd

# parsl dependencies
import parsl
import logging
from parsl.app.app import python_app, bash_app
from parsl.configs.local_threads import Config
from parsl.executors import HighThroughputExecutor # We want to use monitoring, so we must use HTEX
from parsl.executors import MPIExecutor # MPIExecutor wraps the HTEX; need to test if can be used with monitoring
from parsl.monitoring.monitoring import MonitoringHub
from parsl.addresses import address_by_hostname
from parsl.providers import SlurmProvider, LocalProvider

# to display Parsl monitoring GUI in notebook
# Experimental - does not work yet
from IPython.display import IFrame

# To grab near real time solar intensity predictions
from datetime import datetime

# To assign UUID to a workflow run
from uuid import uuid4 as uuid

# Setup tiegcm_utils
package_dir = os.path.abspath('./tiegcm_utils')
# Add the package directory to the Python path
sys.path.append(package_dir)
from tiegcm_utils.tiegcm_inputs import TGCMInput
from tiegcm_utils.perturbed_input_from_indices import write_inp
from tiegcm_utils.fetch_indices import get_f107
from tiegcm_utils.fetch_indices import get_kp_array
#=================================================
# Log everything to stdout (ends up in pink boxes 
# in the notebook). This information is logged anyway
# in ./runinfo/<run_id>/parsl.log. Careful - this has
# the potential to slow the notebook down significantly
# for complex workflows.
# parsl.set_stream_logger() # <-- log everything to stdout
#==================================================

print(parsl.__version__)

2024.12.23


In [175]:
# You can use this cell to cross check the
# ordinal date of year of the date provided.
dt = datetime.strptime(param_datetime_str, '%Y/%m/%d/%H:%M:%S')
print(dt.strftime('%Y'))
print(dt.strftime('%j'))
print(dt.strftime('%H'))

2023
264
00


## Configure Parsl

This configuration must use the `HighThroughputExecutor` (HTEX) since we also want to enable [Parsl monitoring](https://parsl.readthedocs.io/en/latest/userguide/monitoring.html).

In [176]:
config = Config(
    retries=3,
    executors=[
        # Use slurm_htex for running SLURM jobs on the worker nodes
        HighThroughputExecutor(
            label="slurm_htex",
            # cores_per_worker is often more general than cores_per_node
            # and often allows Parsl scale out multiple Parsl-workers on
            # a single worker-node in the most flexible way (i.e. worker
            # nodes of different sizes on different CSPs).
            cores_per_worker=4,
            #max_workers_per_node=3,
            address=address_by_hostname(),
            provider=SlurmProvider(
                partition=param_partition,
                nodes_per_block=1,
                #cores_per_node=4, # Remove this for now - is one core reserved for the Parsl worker?
                init_blocks=2,
                min_blocks=2,
                max_blocks=10,
                exclusive=True,
                worker_init="source "+conda_base_path+"/etc/profile.d/conda.sh; conda activate "+conda_env_name,
                # I don't know why, but Parsl is refusing to relaunch
                # Parsl workers via SLURM. Perhaps the default walltime
                # of 30 mins is a frim, absolute, limit whereas in the
                # past I *think* it has been treated as a per-pilot job limit
                # and new workers would get called up as needed. If this
                # is a firm limit, then Parsl doesn't know that it's happened:
                # once the workers are down, Parsl keeps trying to launch jobs!
                walltime=param_walltime_limit)
        )
    ],
    # If this part of the is not present, no 
    # visualization information is gathered.
    monitoring=MonitoringHub(
        hub_address=address_by_hostname(),
        hub_port=55055,
        monitoring_debug=False,
        resource_monitoring_interval=10,
    ),
    strategy='none'
)

# Loading the configuration starts a Parsl DataFlowKernel
# Pilot jobs are also automatically launched at this point 
# if init_blocks and min_blocks are non-zero. You can
# verify this by checking for files in ./runinfo/<run_id>/submit_scripts/
# as well as monitoring your SLURM allocation with squeue.
dfk = parsl.load(config)

## Define Parsl apps

Parsl workflows are divided into the smallest unit of execution, the app. There are two types of Parsl apps:
1. `python_app`s are useful when launching pure Python code. They are also particularly useful if you want to pass a *small* amount of output from a running app directly back into the workflow. For example, `make_dir` below could be set up as a `python_app` or a `bash_app`. But, I choose to make it a `python_app` because I want the value of `my_dir` to be made explicitly available to the workflow via the `.result()` of the app. This is not possible with a `bash_app`.
2. `bash_app`s are useful when launching tasks on the command line

Here, the applications are *defined* but not run. The `@python_app` and `@bash_app` decorators are the "flags" that tell Parsl that these functions are special and need to be tracked as part of the workflow. Undecorated functions execute locally as regular Python in whatever runtime the notebook is in.

### Python Apps

In [177]:
@python_app # make directory to keep all files associated with the ML model
def make_dir(my_dir):
    import os
    os.makedirs(my_dir, exist_ok = False)
    return my_dir

### Bash Apps

In [178]:
@bash_app # Start the parsl visulaizer
def start_parsl_visualize(
    stdout='parsl_vis_app.stdout', 
    stderr='parsl_vis_app.stderr'):
    return 'parsl-visualize --listen 127.0.0.1 --port 8080'

In [179]:
@bash_app # Get TIEGCM container on-the-fly
def get_container(
    enforce,
    image_dir, 
    container_url, 
    stdout='get_tiegcm_container_app.stdout', 
    stderr='get_tiegcm_container_app.stderr'):
    return '''
    singularity pull {image_dir}/tiegcm.sif {container_url}
    '''.format(
        # I want the image_dir to be a dependency,
        # so it must be a Parsl Future. But the
        # path is stored as the .result() of that
        # Future, hence, the modification here.
        image_dir=image_dir,
        container_url=container_url)

In [180]:
@bash_app # Get data from a bucket and decompress
def get_data(
    enforce,
    destination_dir,
    stdout='get_data_app.stdout', 
    stderr='get_dat_app.stderr'):
    return '''
    #--------------------------
    # Copy from mounted bucket
    cp -v /tiegcm/model-workflows-tiegcm2.0.tar.gz {destination_dir}
    #--------------------------
    # Go to where you copied it
    cd {destination_dir}
    #--------------------------
    # Main decompress
    tar -xzf model-workflows-tiegcm2.0.tar.gz
    #--------------------------
    # Move the decompressed files up one level for shorter paths
    mv -v ./model-workflows-tiegcm2.0/data ./
    mv -v ./model-workflows-tiegcm2.0/model ./
    mv -v ./model-workflows-tiegcm2.0/script ./
    #--------------------------
    # Clean up
    rm -rfv model-workflows-tiegcm2.0
    rm -v model-workflows-tiegcm2.0.tar.gz
    #--------------------------
    # Secondary decompress
    cd data
    tar -xzf tiegcm2.0_res5.0_data.tar.gz
    rm -v tiegcm2.0_res5.0_data.tar.gz
    cd ../
    #--------------------------
    # TODO: Fix this workaround
    touch ./script/truncated_samples_F107.txt
    '''.format(destination_dir=destination_dir)

In [181]:
@bash_app # Setup parameters for ensemble members
def setup_ensemble(
    enforce,
    work_dir="./tiegcm_work",
    ens_size=2,
    mean=75,
    std_dev=1,
    range_limit=5,
    stdout='setup_ensemble_app.stdout', 
    stderr='setup_ensemble_app.stderr'):
    return '''
    #--------------------------
    # Go to work
    cd {work_dir}
    #--------------------------
    # Set some paths
    export TGCMMODEL=`pwd`
    export TGCMDATA=$TGCMMODEL/data/tiegcm_res5.0_data
    #--------------------------
    # Load shared libraries installed using miniconda
    export SINGULARITYENV_LD_LIBRARY_PATH=/opt/miniconda3/lib
    #--------------------------
    # Choose ensemble size and other parameters for F10.7 samples
    ens_size={ens_size}
    mean={mean}
    std_dev={std_dev}
    range_limit={range_limit}
    #--------------------------
    echo "Generate a truncated normal samples for F10.7"
    # Note that some versions of Singularity ASSUME
    # startup in $HOME while others assume startup
    # in $PWD. Need to be explicit with --pwd for 
    # portability. Provide absolute path to running 
    # executable and bind mount it in case.
    # Furthermore, prefix the launch of the container
    # with env -i to force the container to run in
    # a subshell with NO ADDITIONAL ENV VARIABLES.
    # This is critical because otherwise, the container
    # is running in the context of the activated
    # Conda environment used by Parsl workers on the *host*
    # which confuses the Conda in the *container*.
    env -i SINGULARITYENV_LD_LIBRARY_PATH=/opt/miniconda3/lib singularity exec --bind $TGCMMODEL --pwd "$PWD" tiegcm.sif /opt/miniconda3/bin/python $TGCMMODEL/script/generate_F107_samples.py $ens_size $mean $std_dev $range_limit
    '''.format(
        work_dir=work_dir,
        ens_size=ens_size,
        mean=mean,
        std_dev=std_dev,
        range_limit=range_limit
    )

In [182]:
@bash_app # Create and populate ensemble member run dir
def create_run_dir(
    enforce,
    rundir_root="./tiegcm-work",
    runid="42",
    ens_member=1,
    stdout='create_run_dir_app.stdout', 
    stderr='create_run_dir_app.stderr'):
    return '''
    #--------------------------
    # Go to work
    cd {rundir_root}
    #--------------------------
    # Set some paths
    export TGCMMODEL=`pwd`
    #-------------------
    # Setup directory
    #-------------------
    mkdir -p $TGCMMODEL/{runid}/{ens_member}
    cd $TGCMMODEL/{runid}/{ens_member}
    #-------------------
    # Populate
    #-------------------
    /bin/cp -f $TGCMMODEL/script/tiegcm_res5.0.inp .
    /bin/cp -f $TGCMMODEL/truncated_samples_F107.txt .
    /bin/cp -f $TGCMMODEL/script/modify_tgcm_input.py .
    env -i singularity exec --bind $TGCMMODEL --pwd $TGCMMODEL/{runid}/{ens_member} $TGCMMODEL/tiegcm.sif /opt/miniconda3/bin/python modify_tgcm_input.py {ens_member}
    '''.format(
        rundir_root=rundir_root,
        runid=runid,
        ens_member=ens_member)

In [183]:
# This python_app is having trouble finding the custom code
# in tiegcm_utils when running on the remote -> as is, gives
# a "Module not found" error. Flag with _FAILS for now and
# use the bash_app version below that does the same thing.
#@python_app # Create and populate ensemble member run dirs while also perturbing solar forcing and writing namelists.
#def create_ens_dirs_FAILS(
#    enforce,
#    work_dir="./tiegcm-work",
#    runid="42",
#    ens_size=10,
#    kp=0.3,
#    f107=75,
#    f107a=75,
#    stdout='create_ens_dirs_app.stdout', 
#    stderr='create_ens_dirs_app.stderr'):
#    
#    import os
#    import sys
#    package_dir = os.path.abspath('/home/sfgary/tiegcm-digital-twin/tiegcm_utils')
#    # Add the package directory to the Python path
#    sys.path.append(package_dir)
#    from tiegcm_utils.tiegcm_inputs import TGCMInput
#    from tiegcm_utils.perturbed_input_from_indices import write_inp
#    from scipy.stats import norm, truncnorm
#    from datetime import datetime, timedelta
#    
#    # Create the ensemble directories
#    for ii in range(1, ens_size+1):
#        my_dir=f"{work_dir}/run/{runid}/mem{(ii):03}"
#        os.makedirs(my_dir, exist_ok = False)
#    
#    # Launch main function that calls other functions
#    write_inp(N_ENS=ens_size, WORK_DIR=work_dir, JOB_ID=runid, kp=kp, f107=f107, f107a=f107a)
#    
#    output="Done with create_ens_dirs."
#    return output

@bash_app # Create and populate ensemble member run dirs while at the same time perturbing solar forcing. This app combines create_run_dir and setup_ensemble.
def create_ens_dirs(
    enforce,
    work_dir="./tiegcm-work",
    runid="42",
    ens_size=10,
    datetime_str="2023/01/01 16:42:42",
    simulation_hours=24,
    start_type='cold',
    src_file='data/tiegcm_res5.0_data/tiegcm_res5.0_sepeqx_smax_prim.nc',
    stdout='create_ens_dirs_app.stdout', 
    stderr='create_ens_dirs_app.stderr'):
    return '''
    python tiegcm_utils/perturbed_input_from_indices.py {ens_size} {work_dir} {runid} {datetime_str} {simulation_hours} {start_type} {src_file}
    '''.format(
        work_dir=work_dir,
        ens_size=ens_size,
        runid=runid,
        datetime_str=datetime_str,
        simulation_hours=simulation_hours,
        start_type=start_type,
        src_file=src_file
    )
# Unneccessary code from the bash_app
    #--------------------------
    # Get current location
    #start_dir=`pwd`
    #--------------------------
    # Go to work
    #cd {work_dir}
    #--------------------------
    # Set some paths
    #export TGCMMODEL=`pwd`
    #-------------------
    # Perturb and Populate
    #-------------------
    # No need to run from container
    #env -i SINGULARITYENV_LD_LIBRARY_PATH=/opt/miniconda3/lib singularity exec --bind $TGCMMODEL --pwd "$PWD" tiegcm.sif /opt/miniconda3/bin/python $TGCMMODEL/script/perturbed_input_from_indices.py {ens_size} {runid} {kp} {f107} {f107a}
    # Go back to where we started (where the notebook/workflow is running)
    #cd $start_dir
    #--------------------------
    # Create ensemble ensemble dirs
    #ens_size={ens_size}
    #for ((ii = 0; ii < ens_size; ++ii)); do
    #    ii_p_1=$(($ii + 1))
    #    zero_padded_ii_p_1=$(printf "%03d\n" $ii_p_1)
    #    mkdir -p $TGCMMODEL/run/{runid}/mem$zero_padded_ii_p_1
    #done
    #--------------------
    # Add initial conditions file
    # THIS WILL NEED TO BE ADJUSTED WITH WORKFLOW PARAM!
    # Shift this cp operation into perturbed_input_from_indices.py
    # since the src_str is already available there and that is also
    # where we create the ensemble member directories.
    #--------------------
    #for ((ii = 0; ii < {ens_size}; ++ii)); do
    #    ii_p_1=$(($ii + 1))
    #    zero_padded_ii_p_1=$(printf "%03d\n" $ii_p_1)
    #    echo "Currently in $PWD"
    #    echo "running cp $TGCMMODEL/data/tiegcm_res5.0_data/tiegcm_res5.0_mareqx_smin_prim.nc $TGCMMODEL/run/{runid}/mem${{zero_padded_ii_p_1}}/tiegcm_primary_output_src_${{src_str}}.nc"
    #    cp $TGCMMODEL/data/tiegcm_res5.0_data/tiegcm_res5.0_mareqx_smin_prim.nc $TGCMMODEL/run/{runid}/mem${{zero_padded_ii_p_1}}/tiegcm_primary_output_src_${{src_str}}.nc
    #done

In [184]:
@bash_app # Run ensemble member
def run_ens_member(
    enforce,
    rundir_root="./tiegcm-work",
    runid="42",
    ens_member=1,
    stdout='run_ens_member_app.stdout', 
    stderr='run_ens_member_app.stderr'):
    return '''
    #-------------------------
    # Override Parsl SLURM parameter
    # In Parsl HTEX the ntasks-per-node parameter is hardcoded to 1
    # This is why there is a Parsl MPIExecutor. I'm not using it
    # right now so we can check compatibility with older versions
    # of Parsl that don't have the MPIExecutor. FIXME; THIS IS HACK!
    export SLURM_TASKS_PER_NODE="4"
    #--------------------------
    # Go to work
    #--------------------------
    cd {rundir_root}
    #--------------------------
    # Set some paths
    #--------------------------
    export TGCMMODEL=`pwd`
    export TGCMDATA=$TGCMMODEL/data/tiegcm_res5.0_data
    export SINGULARITYENV_LD_LIBRARY_PATH=/opt/miniconda3/lib
    #--------------------------
    # Autodetect name of namelist
    # Error if more than than one
    # is present!
    #--------------------------
    namelist=$(ls -1 $TGCMMODEL/run/{runid}/mem{ens_member}/*.inp)
    num_inp=`echo $namelist | awk '{{print NF}}' `
    if [ $num_inp -lt 1 ]; then
        echo "Error More than one namelist (.inp) found!"
        exit 1
    fi
    #--------------------------
    # Launch
    #--------------------------
    # Originally, it was assumed that this container
    # ran somewhere under $HOME whose absolute path 
    # is the same in the container and out of the 
    # container and Singularity autobinds $HOME on
    # startup of the container. When choosing other
    # directories not under $HOME (or the limited
    # subset of autobound dirs), however, we need to 
    # explicitly include the --bind directive.
    source $HOME/ompi/env.sh
    time mpirun -n 4 singularity exec --bind $TGCMMODEL --pwd $TGCMMODEL/run/{runid}/mem{ens_member} $TGCMMODEL/tiegcm.sif /opt/model/tiegcm.exec/tiegcm2.0 $namelist &> $TGCMMODEL/run/{runid}/mem{ens_member}/tiegcm_res5.0_{ens_member}.out
    #--------------------------
    # Export outputs to s3 bucket
    #--------------------------
    # DONT DO THIS HERE! MAKE A SEPARATE APP THAT
    # SIMPLY RSYNCS ALL ENSEMBLE MEMBER SUBDIRS
    # BY RSYNCING RUNID DIR. ALSO, THIS NEW APP
    # SHOULD DEPEND ON ALL (? or 80%?) OF THE 
    # PREVIOUS ENSEMBLE MEMBERS AND CLEAN OUT
    # THE LARGE NETCDF FILES (e.g. line 13 in clean.sh).
    #archive_dir="/tiegcm/model-outputs/tiegcm/tiegcm2.0/ens/{runid}/{ens_member}"
    #mkdir -p $archive_dir
    #rsync -r {rundir_root}/run/{runid}/mem{ens_member}/. /tiegcm/model-outputs/tiegcm/tiegcm2.0/ens/{runid}/{ens_member}
    '''.format(
        rundir_root=rundir_root,
        runid=runid,
        ens_member=f"{(ens_member):03}")

## Start Parsl monitoring - Option 1 - direct shell invocation to background

This step can be done at any point provided that a database file exists.  The default location of this file is in `./runinfo/monitoring.db` and this file is created when the Parsl configuration is loaded. When the notebook kernel is restarted, additional Parsl workflow runs' information is appended to the monitoring information in `./runinfo`. It is possible to view this information "offline" (i.e. no active running Parsl workflows) and also fully independently of this notebook (see Option 3, at the end of this notebook for how to specify custom locations fort the monitoring DB). For our purposes here, we will assume that we're using a monitoring DB in the same directory as the notebook runtime in `./runinfo`.

This launch can be commented out here since it is also possible to launch `parsl-visualize` from a Parsl app within the workflow, examples of which are below. The advantage to running `parsl-visualize` as a Parsl app is that the visualization server is up and running while the workflow is running and then is shut down when the workflow is cleaned up. Otherwise, when `parsl-visualize` is launched via `os.system` the running child process can persist even after workflow shut down or notebook kernel restart. Here, however, we opt not to use `parsl-visualize` in the workflow because we want only one executor for this workflow for simplicity (to send jobs to the SLURM scheduler) but `parsl-visualize` would need to be started on a local (to the head node) executor.

You can rerun this command even if `parsl-visualize` is already running because it checks for existing port usage and if that port is already in use, it fails silently here (on the command line it will give you an error).

In [185]:
# Launch Parsl visualizer
os.system('parsl-visualize 1> parsl_vis.stdout 2> parsl_vis.stderr &')

0

## Parsl Workflow starts here

### Make a directory for workflow app logs

As part of the workflow, we make a directory for all the app logs. It looks like the first Parsl app to be invoked will start the Parsl interchanges and pilot jobs. This means that you may need to wait some time for this first app to start if you have a queue/cloud spinup wait time associated with getting an allocation of worker nodes **even if this first app is NOT going to worker nodes**!

Note that the use of `*_future.result()` blocks the notebook execution (i.e. the cell stays in pending state) until the result of the app future is realized. You can, however, access the state of the future with `*_future` without blocking the workflow. If you don't invoke `*_future.result()`, then the notebook execution contines and builds out the whole Parsl DAG until you reach the first blocking cell.

In [186]:
# Launch the app
make_log_dir_future = make_dir(param_log_dir)

# Get the result
make_log_dir_future.result()
log_dir = param_log_dir

### Start Parsl monitoring - Option 2 - Monitoring as an app within a Parsl workflow

This approach is helpful if we want Parsl Monitoring processes to be cleaned up after the workflow is complete. Note that this command is tracked by Parsl and is considered to be part of the workflow. Since we defined the app to use the `local_htex` above, it is running on the head node of the cluster. You can verify this placement in the terminal with `ps -u $USER -HF -ww | grep vis`.

In [187]:
# Start Parsl visualization in a
# separate cell since we only want
# to run this app one time.
#parsl_vis_future = start_parsl_visualize(
    #make_log_dir_future, 
#    stdout=log_dir+'/parsl_vis_app.stdout', 
#    stderr=log_dir+'/parsl_vis_app.stderr')

In [188]:
# I'd love to view Parsl monitoring in the notebook,
# but this doesn't work.
# IFrame('http://localhost:8080', width=600, height=500)

### Set up the TIEGCM ensemble

1. Set up an overarching `work_dir`.
2. Get TIEGCM container
3. Grab TIEGCM initialization data and run scripts. They are stored on a shared bucket and we are allowed to redistribute for non-commercial and academic purposes provided that we acknowledge with a [link to the original license](https://www.hao.ucar.edu/modeling/tgcm/tiegcm2.0/release/html/_downloads/tiegcmlicense.txt).
4. Establish working directories/parameters for each ensemble member. 

In [189]:
make_work_dir_future = make_dir(param_work_dir_root)
work_dir=make_work_dir_future.result()time.sleep(4)

In [190]:
# Get the container as a Parsl App
get_container_as_parsl_app=False
# This works, but it is *very* slow for a cluster without (expensive) 
# high performance storage because the job runs on a worker node
# which attempts to write the ~2GB container to an NFS-mounted
# shared drive that is on the head node. It's much faster to just
# run the singularity pull command on the head node. The alternative
# is to set up two executors, but that is not done here.
if (get_container_as_parsl_app):
    get_container_future = get_container(
        enforce=make_work_dir_future,
        image_dir=work_dir, 
        container_url=param_container_url, 
        stdout=log_dir+'/get_tiegcm_container_app.stdout', 
        stderr=log_dir+'/get_tiegcm_container_app.stderr')
else:
    # Local run of singularity pull (i.e. on head node)
    # Note removal of & compared to similar local launch of 
    # a process with parsl-visualize (running in the background).
    # Here, we want the singularity pull to block execution
    # until finished.
    os.system('sleep 4 ; singularity pull '+work_dir+'/tiegcm.sif '+param_container_url+' 1> singularity_pull.stdout 2> singularity_pull.stderr')

### Get background data, initializations, and scripts

In [191]:
get_data_future = get_data(
    make_work_dir_future, 
    work_dir,
    stdout=log_dir+'/get_data_app.stdout', 
    stderr=log_dir+'/get_data_app.stderr')

### Create ensemble parameters
This step grabs near real time solar forcing data based on the date and generates an uncertainty range of these parameters which is used as a basis for the spread of the ensemble members.

In [192]:
# Built into create_ens_dirs
#dt = datetime.strptime(param_datetime_str, '%Y/%m/%d %H:%M:%S')
#(f107, f107a) = get_f107(dt)
#kp_array = get_kp_array(dt)

# Older app below which creates the distribution
# of F107 (only) independent of the setup of
# ensemble directories.
#setup_ensemble_future = setup_ensemble(
#    get_data_future,
#    work_dir=work_dir,
#    ens_size=param_ens_size,
#    mean=f107_mean,
#    std_dev=f107_std_dev,
#    range_limit=param_range_limit,
#    stdout=log_dir+'/setup_ensemble_app.stdout', 
#    stderr=log_dir+'/setup_ensemble_app.stderr')

### Set up run dirs for each ensemble member
This step also creates/modifies the namelist used to define/configure the TIEGCM app. This namelist
also needs to know the date and filenames to read/write to in addition to the solar forcing parameters,
ensemble size, and where it is running.

In [193]:
# Create a unique ID for this run
runid=uuid()

# Older app below with creates ensemble
# directories independent of the perturbation
# of solar forcing.
#create_run_dir_future_list=[]
#for ii in range(1, param_ens_size+1):
#    create_run_dir_future_list.append(
#        create_run_dir(
#            enforce=setup_ensemble_future,
#            rundir_root=work_dir,
#            runid=runid,
#            ens_member=ii,
#            stdout=log_dir+'/create_run_dir_app.stdout', 
#            stderr=log_dir+'/create_run_dir_app.stderr'))

create_ens_dirs_future = create_ens_dirs(
    get_data_future,
    work_dir=work_dir,
    runid=runid,
    ens_size=param_ens_size,
    datetime_str=param_datetime_str,
    simulation_hours=param_simulation_hours,
    start_type=param_start_type,
    src_file=param_src_file,
    stdout=log_dir+'/create_ens_dirs_app.stdout', 
    stderr=log_dir+'/create_ens_dirs_app.stderr')

#======================================================
# Do it outside of a Parsl app
#for ii in range(1, param_ens_size+1):
#    my_dir=f"{work_dir}/run/{runid}/mem{(ii):03}"
#    os.makedirs(my_dir, exist_ok = False)
#        
#write_inp(N_ENS=param_ens_size, WORK_DIR=work_dir, JOB_ID=runid, kp=kp_array[-1], f107=f107, f107a=f107a)
#======================================================

In [194]:
create_ens_dirs_future.__dict__

{'_condition': <Condition(<unlocked _thread.RLock object owner=0 count=0 at 0x145eb58576f0>, 0)>,
 '_state': 'PENDING',
 '_result': None,
 '_exception': None,
 '_waiters': [],
 '_done_callbacks': [functools.partial(<bound method DataFlowKernel.handle_app_update of <parsl.dataflow.dflow.DataFlowKernel object at 0x145eb5857070>>, {'args': (<AppFuture at 0x145eb58577c0 state=pending>,), 'depends': [<AppFuture at 0x145eb58577c0 state=pending>], 'dfk': <parsl.dataflow.dflow.DataFlowKernel object at 0x145eb5857070>, 'executor': 'slurm_htex', 'func': <function wrap_error.<locals>.wrapper at 0x145eb58a6ee0>, 'func_name': 'create_ens_dirs', 'kwargs': {'work_dir': './tiegcm-work', 'runid': UUID('55afcfdb-821f-443a-abf3-78391c0d5713'), 'ens_size': 10, 'datetime_str': '2023/09/21/00:00:00', 'simulation_hours': 2, 'start_type': 'cold', 'src_file': 'data/tiegcm_res5.0_data/tiegcm_res5.0_sepeqx_smax_prim.nc', 'stdout': './parsl-app-logs/create_ens_dirs_app.stdout', 'stderr': './parsl-app-logs/create_

### Run the TIEGCM ensemble

Launch each ensemble member; this step is dependent on the creation of the ensemble member working directories

In [195]:
# Older app for when creation of ensemble dir
# are separate parsl apps.
#run_ens_member_future_list=[]
#for ii in range(1, param_ens_size+1):
#    run_ens_member_future_list.append(
#        run_ens_member(
#            enforce=create_run_dir_future_list[ii-1],
#            rundir_root=work_dir,
#            runid=runid,
#            ens_member=ii,
#            stdout=log_dir+'/run_ens_member_app.stdout', 
#            stderr=log_dir+'/run_ens_member_app.stderr'))

run_ens_member_future_list=[]
for ii in range(1, param_ens_size+1):
    run_ens_member_future_list.append(
        run_ens_member(
            enforce=create_ens_dirs_future,
            rundir_root=work_dir,
            runid=runid,
            ens_member=ii,
            stdout=log_dir+'/run_ens_member_app.stdout', 
            stderr=log_dir+'/run_ens_member_app.stderr'))

In [196]:
run_ens_member_future_list[1].result()  # Wait until the future completes

0

## Stop Parsl

The cells above can be rerun any number of times; this will simply send more and more apps to be run by Parsl. When the workflow is truly complete, it is time to call the cleanup() command. This command runs implicitly when a `main.py` script finishes executing, but it is *not* run in a notebook unless it is explicitly called as it is below.

In [197]:
for future in run_ens_member_future_list:
    future.result()
dfk.cleanup()

## Clean up Parsl log files

In [30]:
# This directory contains Parsl monitoring logs
#! rm -rf runinfo

# This directory contains the Parsl app logs
#! rm -rf {log_dir}

# This is the working directory
#! rm -rf {work_dir}

# Shut down parsl-visualize
#! killall parsl-visualize

## Start Parsl Monitoring - Option 3 - Post workflow manual invocation

Once the Parsl `./runinfo/monitoring.db` is created, it is possible to start Parsl Monitoring and browse the results of workflow in an offline manner.  In this scenario, `parsl-visualize` can be started on the command line provided that a Conda env with `parsl[visualize]` installed is activated. For example:
```
source pw/.miniconda3/etc/profile.d/conda.sh
conda activate base
parsl-visualize sqlite:////${HOME}/<work_dir>/runinfo/monitoring.db
```
(You may need to adjust the path to the Conda environment, its name, and the path to `monitoring.db`.)